# Appendices

Kept out of `quantile_learning.ipynb` because none of this is part of the
experiment: these are the three demonstrations that delimit the hypotheses,
and they are run once and quoted, not swept.

* **A.1** contractivity cannot simply be dropped: rescale the state weights of
  a fitted map and watch uniqueness fail.
* **A.2** a recursion with provably two solutions, so that the failure in A.1
  is not an artefact of the fit.
* **B** a chaotic drift with stochastic forcing: the contraction condition is
  sufficient and, on this family, short of necessary by about a factor of
  three; the operative boundary is a negative Lyapunov exponent.

> **A.1 and A.2 were rebuilt from the uniqueness remark of the draft** rather
> than copied from the original notebook, which was not reachable when this
> package was assembled.  The mathematical content is the remark's: the two
> counterexamples are $\hat q(u;z)=\Pi_{\mathcal X}(2z_1)$ and
> $\hat q(u;z)=-z_1$.  Worth diffing against the originals.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))
import config as CFG
torch.set_num_threads(getattr(CFG, "TORCH_THREADS", 2))

from generators.synthetic_generators import build_target
from generators.quantile_generator import ClosedQuantileRecursion
from generators.simulate_paths import simulate, make_windows, initial_spread
from fit_generator import fit
from utils.hypotheses import lipschitz_empirical, theta_hat
from utils.marginal import w1_marginal
from utils.autocorrelation import acf
from utils.lyapunov import lyapunov, lyapunov_decomposition
from utils.plotting import C_TARGET, C_GEN, C_BASE, C_RULE, C_BOUND

M = 1.0

## Appendix A. The contraction condition is sufficient, not necessary

The generated law is pinned to within $2\theta/(1-S)$ by the comparison
argument alone, which uses nothing about $\hat q$ beyond $\theta$.  What
contractivity of $\hat q$ buys is not the bound but everything the *adapted*
conclusion needs, and it buys it through uniqueness of the solution of the
closed recursion:

* a measurable selection from a multivalued solution set may read the whole
  innovation path, so innovation-adaptedness fails, and with it bicausality;
* the solution correspondence is shift equivariant while a measurable
  selection of it need not be, so stationarity fails.

Smallness of $\theta$ does not help:
$\Pi_{\mathcal X}(q + \theta\sin(z_1/\lambda))$ is within $\theta$ of $q$ and
has contraction constant of order $\theta/\lambda$.

### A.1 Rescale the state weights, and watch uniqueness fail

In [ ]:
# Fit a small net on a contractive target, then scale G by a growing factor.
# Scaling G multiplies every lag sensitivity, so sum_i Lip_i grows while the
# map stays continuous and X-valued.  The question is when the closed
# recursion stops forgetting where it started.

target = build_target("linear", 0.65, CFG.TARGET_SPECIFICS["linear"], M=M)
path = simulate(target, 20000, burn=2000, seed=1)
Z, Xs = make_windows(path, 2)
net0, _ = fit(Z, Xs, 2, width=96, epochs=60, lip_mode="penalty",
              pen_region="hood", L_max=0.95, lam=1.0, tail_frac=0.2, M=M)

import copy
rows = []
for f in (1.0, 1.4, 1.8, 2.2, 2.6, 3.0, 4.0):
    net = copy.deepcopy(net0)
    net.rescale_G(f)
    rec = ClosedQuantileRecursion(net)
    lip = float(lipschitz_empirical(net, np.random.default_rng(0).uniform(-M, M, (2000, 2))).sum())
    _, spread = initial_spread(rec, n_steps=120, n_starts=24, M=M)
    gen = simulate(rec, 20000, burn=2000, seed=404)
    lam_val, _ = lyapunov(net, gen)
    rows.append(dict(factor=f, lip=lip, lam=lam_val,
                     sp40=spread[40], sp119=spread[-1], sd=float(gen.std())))
    print(f"  G x{f:<4.1f}  sum_i Lip_i {lip:6.3f}   lambda {lam_val:+7.3f}   "
          f"spread t=40 {spread[40]:.2e}   t=119 {spread[-1]:.2e}   sd(gen) {gen.std():.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
f = [r["factor"] for r in rows]
ax[0].plot(f, [r["lip"] for r in rows], "o-", color=C_GEN, label=r"$\sum_i$Lip$_i(\hat q)$")
ax[0].plot(f, [r["lam"] for r in rows], "s--", color=C_BASE, label=r"$\lambda$")
ax[0].axhline(1, color=C_BOUND, ls=":", lw=1); ax[0].axhline(0, color=C_RULE, lw=.8)
ax[0].set_xlabel("factor applied to G"); ax[0].legend(fontsize=8, frameon=False)
ax[0].set_title("the fitted map is pushed out of contraction")

ax[1].semilogy(f, [max(r["sp119"], 1e-18) for r in rows], "o-", color=C_GEN)
ax[1].set_xlabel("factor applied to G")
ax[1].set_title("spread across 24 starts at t = 119")
plt.tight_layout(); plt.show()

What to read.  While $\sum_i\mathrm{Lip}_i<1$ the spread collapses to machine
zero: the closed recursion has one solution and the generator is a function of
the innovations alone.  Past it the spread survives, and the object the
recursion defines is no longer a single stationary process.  Note also that
the collapse tracks $\lambda$ rather than $\sum_i\mathrm{Lip}_i$ crossing one,
which is the same phenomenon Appendix B sweeps deliberately.

### A.2 A recursion with provably two solutions

In [ ]:
# Two maps from the uniqueness remark, both continuous and X-valued, neither
# contractive.  No fitting involved: the failure is in the map, not the fit.
#
#   (i)  qhat(u;z) = Pi_X(2 z_1).   The constants 0 and +/-M all solve the
#        closed recursion, so a selection may be made by reading u_0.
#   (ii) qhat(u;z) = -z_1.          The solutions are x_t = (-1)^t a for any
#        |a| <= M, so the solution correspondence is shift equivariant while a
#        selection of it need not be, and stationarity fails.

def run(qfun, z0, n=40, seed=0):
    u = np.random.default_rng(seed).random(n)
    x, z = np.empty(n), float(z0)
    for t in range(n):
        x[t] = qfun(u[t], z); z = x[t]
    return x

q_doubling = lambda u, z: float(np.clip(2.0 * z, -M, M))
q_flip     = lambda u, z: float(-z)

print("(i)  qhat(u;z) = Pi_X(2 z_1):  three distinct constant solutions")
for z0 in (0.0, M, -M):
    x = run(q_doubling, z0)
    print(f"     start {z0:+.1f}  ->  x_0..x_4 {np.round(x[:5], 6).tolist()}   "
          f"constant: {np.allclose(x, x[0])}")

print("\n(ii) qhat(u;z) = -z_1:  a two-cycle through every |a| <= M")
for z0 in (0.3, -0.7):
    x = run(q_flip, z0)
    print(f"     start {z0:+.1f}  ->  x_0..x_5 {np.round(x[:6], 6).tolist()}   "
          f"period 2: {np.allclose(x[::2], x[0]) and np.allclose(x[1::2], x[1])}")

print("\n     the SAME innovation stream drives all of these, so the solution")
print("     selected is not a function of the innovations: adaptedness, and")
print("     with it bicausality, fails before any approximation error enters.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for z0, col in ((0.0, C_TARGET), (M, C_GEN), (-M, C_BASE)):
    ax[0].plot(run(q_doubling, z0)[:20], "o-", ms=3, color=col, label=f"start {z0:+.1f}")
ax[0].set_title(r"(i) $\hat q(u;z)=\Pi_X(2z_1)$: three solutions")
ax[0].set_xlabel("t"); ax[0].legend(fontsize=8, frameon=False)
for z0, col in ((0.3, C_TARGET), (-0.7, C_GEN)):
    ax[1].plot(run(q_flip, z0)[:20], "o-", ms=3, color=col, label=f"start {z0:+.1f}")
ax[1].set_title(r"(ii) $\hat q(u;z)=-z_1$: a two-cycle for every start")
ax[1].set_xlabel("t"); ax[1].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

### What the two experiments say

A.1 shows the failure arriving continuously in a fitted object, and A.2 shows
it is not an artefact of fitting: two elementary continuous $\mathcal X$-valued
maps already have multiple solutions.  Together they say that contractivity in
the generator proposition is not decoration.  What it is *not* is necessary:
Appendix B shows the pathwise and adapted conclusions surviving well past
$S=1$, with the real boundary at $\lambda = 0$.

## Appendix B. A chaotic drift with stochastic forcing

Take the logistic map at $r$, carry it to $[-M,M]$, centre and rescale it so
its range covers a fraction $1-\rho$ of the interval, leaving $\rho$ for the
innovation:

$$m_0(x) = \frac{r(x+M)(M-x)}{2M} - M,\qquad \sup_{|x|\le M}|m_0'| = r,$$

$$m(x) = \gamma\,(m_0(x) - \mathrm{mid}),\quad \gamma = \frac{4(1-\rho)}{r},
\qquad X_t = m(X_{t-1}) + \rho M\,g(u_t).$$

Since $m_0$ ranges over an interval of width $Mr/2$,

$$\boxed{\;\ell_1 = \sup_{u,z}\Bigl|\frac{\partial q}{\partial z}\Bigr| = 4(1-\rho)\;}$$

**independently of $r$.**  So one knob moves the target across the standing
hypothesis, which fails for $\rho<3/4$, and keeps going, while $r$ stays pinned
in the chaotic regime.  In this package that knob is $S$ itself:
$\rho = 1 - S/4$.

The target is 1-Markov, bounded, and has a well-defined stationary law at every
$\rho$ tested.  What it does not have below $\rho=3/4$ is a uniform
contraction, so the bound $\theta/(1-S)$ is unavailable.

**The Lipschitz penalty must be off.**  At `L_max = 0.95` it would force the
learned map to be contractive, and a contractive map cannot approximate a
target with $S=3.4$.  `run_one` does this automatically; here it is explicit.
Fitting at $m=1$, the target's own order, makes the reported exponent the exact
top Lyapunov exponent rather than an upper bound.

In [ ]:
S_LIST = [0.80, 2.00, 2.80, 3.40]        # rho = 0.80, 0.50, 0.30, 0.15
R_LOG, T_B, EPOCHS_B = 3.7, 40000, 120

res_b = []
for S in S_LIST:
    t = build_target("logistic", S, dict(r=R_LOG), M=M)
    p = simulate(t, T_B, burn=2000, seed=1)
    Z, X = make_windows(p, 1)
    net, _ = fit(Z, X, 1, width=256, epochs=EPOCHS_B, lip_mode="none",
                 tail_frac=0.2, M=M, seed_net=0)
    rec = ClosedQuantileRecursion(net)

    tru = simulate(t,   T_B, burn=2000, seed=303)
    gen = simulate(rec, T_B, burn=2000, seed=404)
    us  = np.random.default_rng(7).random(22000)
    xt  = simulate(t,   20000, burn=2000, seed=0, u_stream=us)
    xh  = simulate(rec, 20000, burn=2000, seed=0, u_stream=us)
    _, spread = initial_spread(rec, n_steps=80, n_starts=24, M=M)
    lam_val, exact = lyapunov(net, gen)
    lip = float(lipschitz_empirical(net, np.random.default_rng(0).uniform(-M, M, (3000, 1))).sum())
    th  = theta_hat(net, t, "data", Z_data=Z, m=1)

    res_b.append(dict(S=t.S, rho=t.rho, target=t, net=net, tru=tru, gen=gen,
                      err=np.abs(xt - xh), spread=spread, lam=lam_val, exact=exact,
                      lip=lip, theta=th, w1=w1_marginal(tru, gen),
                      sd=float(tru.std()), acf_t=acf(tru, 5)[1:], acf_g=acf(gen, 5)[1:],
                      decomp=lyapunov_decomposition(t, gen)))
    r = res_b[-1]
    print(f"rho={t.rho:.2f}  S={t.S:.2f}  lip_hat {lip:5.2f}  lambda {lam_val:+6.3f}"
          f"  theta {th:.3f}  sup|X-Xh| {r['err'].max():7.3f}"
          f"  spread40 {spread[40]:.2e}  W1/sd {r['w1']/r['sd']:.4f}")

In [ ]:
print(f"{'rho':>5}{'S':>7}{'lip_hat':>9}{'lambda':>9}{'theta':>8}"
      f"{'theta/(1-S)':>13}{'sup|X-Xh|':>11}{'spread40':>11}{'W1/sd':>8}{'ACF1 t/g':>18}")
for r in res_b:
    b = f"{r['theta']/(1-r['S']):.3f}" if r["S"] < 1 else "void"
    print(f"{r['rho']:5.2f}{r['S']:7.2f}{r['lip']:9.2f}{r['lam']:+9.3f}{r['theta']:8.3f}"
          f"{b:>13}{r['err'].max():11.3f}{r['spread'][40]:11.2e}"
          f"{r['w1']/r['sd']:8.4f}   {r['acf_t'][0]:+.3f} / {r['acf_g'][0]:+.3f}")

print("\nthe TARGET's exponent, decomposed along the generated path:")
print("  lambda = log 4(1-rho) + E log|X|/M .  The second term is strictly negative")
print("  because |X| <= M, which is exactly why S = 4(1-rho) can exceed 1 while")
print("  lambda stays negative.  The autograd column is the LEARNED map's exponent;")
print("  the two agree once the fit has converged, and a gap means it has not.")
for r in res_b:
    d = r["decomp"]
    print(f"  rho={r['rho']:.2f}   {d['log_sup_slope']:+.4f} {d['E_log_abs_X_over_M']:+.4f}"
          f"  = {d['lambda_analytic']:+.4f} (target)   vs {r['lam']:+.4f} (learned)"
          f"   geometric mean |X| = {d['geometric_mean_abs_X']:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
rh = np.array([r["rho"] for r in res_b])

ax[0].plot(rh, [r["S"] for r in res_b], "o-", color=C_TARGET, label=r"$S = 4(1-\rho)$ (target)")
ax[0].plot(rh, [r["lip"] for r in res_b], "s--", color=C_GEN, label=r"$\sum_i$Lip$_i(\hat q)$")
ax[0].axhline(1, color=C_BOUND, ls=":", lw=1)
ax[0].set_xlabel(r"noise budget $\rho$"); ax[0].invert_xaxis()
ax[0].legend(fontsize=8, frameon=False)
ax[0].set_title("the fit inherits S: nothing in the\nestimator prefers a contraction", fontsize=9)

ax[1].plot(rh, [r["lam"] for r in res_b], "o-", color=C_BASE)
ax[1].axhline(0, color=C_BOUND, ls=":", lw=1)
ax[1].set_xlabel(r"$\rho$"); ax[1].invert_xaxis()
ax[1].set_title(r"top Lyapunov exponent $\lambda$", fontsize=9)

ax[2].semilogy(rh, [max(r["err"].max(), 1e-14) for r in res_b], "o-", color=C_GEN,
               label=r"$\sup_t|X_t-\hat X_t|$ (shared)")
ax[2].semilogy(rh, [max(r["spread"][40], 1e-16) for r in res_b], "v-.", color=C_RULE,
               label="spread across 24 starts")
ax[2].semilogy(rh, [r["w1"] / r["sd"] for r in res_b], "s--", color=C_TARGET,
               label=r"$W_1$/sd (independent)")
ax[2].set_xlabel(r"$\rho$"); ax[2].invert_xaxis(); ax[2].legend(fontsize=8, frameon=False)
ax[2].set_title("the pathwise statement dies,\nthe law does not", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# the chaotic end, in detail: a spiky invariant density recovered exactly while
# the pathwise coupling has completely failed
r = res_b[-1]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
bb = np.linspace(-M, M, 120)
ax[0].hist(r["tru"], bins=bb, density=True, alpha=.55, color=C_TARGET, label="target")
ax[0].hist(r["gen"], bins=bb, density=True, alpha=.55, color=C_GEN, label="generated")
ax[0].set_title(fr"invariant density, $\rho$={r['rho']:.2f}, $S$={r['S']:.1f}", fontsize=10)
ax[0].legend(fontsize=8, frameon=False)

ax[1].plot(np.r_[1:6], r["acf_t"], "o-", color=C_TARGET, ms=4, label="target")
ax[1].plot(np.r_[1:6], r["acf_g"], "s--", color=C_GEN, ms=4, label="generated")
ax[1].axhline(0, color=C_RULE, lw=.8); ax[1].set_xlabel("lag")
ax[1].set_title("autocorrelation", fontsize=10); ax[1].legend(fontsize=8, frameon=False)

ax[2].semilogy(np.maximum(r["err"][:400], 1e-12), lw=.7, color=C_GEN)
ax[2].set_xlabel("t"); ax[2].set_title(r"$|X_t-\hat X_t|$, shared innovations", fontsize=10)
plt.tight_layout(); plt.show()

### The three regimes

| | hypothesis | conclusion |
|---|---|---|
| $S<1$ | holds | $\theta/(1-S)$ valid, and slack by roughly an order of magnitude |
| $S\ge1$, $\lambda<0$ | fails | pathwise closeness survives anyway; synchronisation still occurs |
| $\lambda>0$ | fails | the pathwise statement is void; the **law** is still recovered |

So on this family $\sum_i\ell_i<1$ is sufficient and short of necessary by
about a factor of three: everything keeps working to $S\approx2.8$.  The
operative boundary for the pathwise and adapted statements is $\lambda<0$,
which is the criterion the stochastic echo state property literature uses.

Two further readings.

**Nothing in the estimator prefers a contraction.**  With the penalty off, the
fitted $\sum_i\mathrm{Lip}_i$ tracks the target's $S$ to within a few per cent
at every point, $S=3.4$ included.  Whatever contraction the main experiment's
fits exhibit came from the penalty and from the target, not from the method.

**The failure is entirely in the amplification.**  $\theta$ is essentially flat
across the sweep: the approximation does not degrade as the target becomes
chaotic.  It is $1/(1-S)$ that breaks, which is the cleanest statement of what
the contraction hypothesis is doing in the theorem.  It is not there to make
the target learnable; it is there to stop the error compounding.

**Reading it against the main notebook.**  Section 10 there exhibits a
generator that passes the path-law diagnostics while getting the conditional
law wrong.  This appendix is the other half: a generator that gets the law
right, marginals, tails and autocorrelation, while the adapted statement is
void.  The pair is why the adapted distance is the certificate and the path
law is not.